# Week 2: the toy detector and the data pipeline

We generate the dataset used for the rest of the course, look at events, and check that the
`ToyTrackingDataset` you wrote produces sensible tensors.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

In [ ]:
data_dir = ensure_toy_data(num_train=200 if FAST else 2000, num_val=50 if FAST else 200, num_test=50 if FAST else 200)
print(data_dir, {s: len(list((data_dir / s).glob("*-hits.parquet"))) for s in ("train", "val", "test")})

## 1. One raw event

Every hit has a `particle_id`; 0 means noise. Colours below are particles. Look at the transverse view:
low-pT particles curve more (radius `R = pT / (0.3 B)`). In the r-z view tracks are straight lines.

In [ ]:
import pandas as pd

from minihep.plotting import event_display

hits = pd.read_parquet(data_dir / "train" / "event000000001-hits.parquet")
parts = pd.read_parquet(data_dir / "train" / "event000000001-parts.parquet")
print(hits.head(), "\n", parts.head())
event_display(hits.x / 1000, hits.y / 1000, hits.z / 1000, hits.particle_id, title="event 1 (all particles)")
plt.show()

In [ ]:
parts_all = pd.concat([pd.read_parquet(p) for p in sorted((data_dir / "train").glob("*-parts.parquet"))[:200]])
pt = np.hypot(parts_all.px, parts_all.py)
eta = np.arcsinh(parts_all.pz / pt)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
axes[0].hist(pt, bins=60, range=(0, 10))
axes[0].set_xlabel("pT [GeV]")
axes[0].set_yscale("log")
axes[1].hist(eta, bins=60)
axes[1].set_xlabel("eta")
axes[2].hist(parts_all.nhits, bins=np.arange(10) - 0.5)
axes[2].set_xlabel("hits per particle")
axes[3].hist([len(pd.read_parquet(p)) for p in sorted((data_dir / "train").glob("*-hits.parquet"))[:200]], bins=30)
axes[3].set_xlabel("hits per event")
plt.tight_layout()
plt.show()

## 2. Through your Dataset

`dataset[i]` returns `(inputs, targets)`. Check every shape against the docstring of `minihep/data/dataset.py`.

In [ ]:
from minihep.data.dataset import ToyTrackingDataset

inputs_cfg = {"hit": ["x", "y", "z", "r", "eta", "phi"]}
targets_cfg = {"hit": ["on_valid_particle", "is_first"], "particle": ["pt", "eta", "phi"]}
ds = ToyTrackingDataset(str(data_dir / "train"), inputs_cfg, targets_cfg, event_max_num_particles=32)
inputs, targets = ds[0]
for name, d in (("inputs", inputs), ("targets", targets)):
    for k, v in d.items():
        print(f"{name:8s} {k:24s} {tuple(v.shape)!s:14s} {v.dtype}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
order = torch.argsort(inputs["hit_phi"][0])
axes[0].imshow(targets["particle_hit_valid"][0][:, order], aspect="auto", interpolation="nearest", cmap="Greys")
axes[0].set_xlabel("hit (sorted by phi)")
axes[0].set_ylabel("particle slot")
axes[0].set_title("particle_hit_valid: the mask targets")
hits_v, parts_v = ds.load_event(0)
labels = np.where(hits_v.on_valid_particle, hits_v.particle_id, 0)
event_display(hits_v.x, hits_v.y, hits_v.z, labels, title="reconstructable particles only (pT > 1 GeV)", axes=axes[1:])
plt.tight_layout()
plt.show()

## 3. Parquet vs HDF5

Pack the training split into one HDF5 file and compare size and read speed.

In [ ]:
import time

import h5py

from minihep.data.prep import parquet_dir_to_hdf5

h5_path = data_dir / "train.h5"
if not h5_path.exists():
    parquet_dir_to_hdf5(data_dir / "train", h5_path)
size_pq = sum(p.stat().st_size for p in (data_dir / "train").glob("*.parquet")) / 1e6
print(f"parquet: {size_pq:.1f} MB   hdf5 (lzf): {h5_path.stat().st_size / 1e6:.1f} MB")
with h5py.File(h5_path) as f:
    name = sorted(f)[0]
    f[name].visititems(lambda path, obj: print(f"  {name}/{path}", getattr(obj, "shape", ""), getattr(obj, "dtype", "")))

for source in (str(data_dir / "train"), str(h5_path)):
    d = ToyTrackingDataset(source, inputs_cfg, targets_cfg)
    t0 = time.perf_counter()
    [d[i] for i in range(min(100, len(d)))]
    print(f"{Path(source).name:10s} {(time.perf_counter() - t0) * 10:.2f} ms/event")

## Your turn

1. Change `particle_min_pt` to 0.5 and see how the number of target particles per event changes.
2. Make `pad_collate` batch 4 events and `imshow` the padded `hit_valid` mask.
3. In hepattn, open `experiments/trackml/data.py` and find the line that casts inputs with `.half()`.
   Which TrackML input in your vault notes overflowed because of it? (Week 8 returns to this.)